# Corpus Scoring

This notebook starts the reproducible local-model scoring workflow for earnings-call transcripts.

It is intentionally notebook-first: we test transcript batching, prompt design, Hugging Face model loading, and output parsing here before polishing the workflow into `.py` modules.

## 1. Setup & Environment

The first cells confirm the Python environment, CUDA/GPU visibility, project paths, and import the reusable model loader from `src.models`. Hugging Face models are cached under the project-level `models/` folder.

In [ ]:
# import core libraries and print environment details
import json
import platform
import sys
import textwrap
from pathlib import Path

import pandas as pd
import torch


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "data" / "transcripts" / "transcripts.parquet").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root()
TRANSCRIPTS_PATH = PROJECT_ROOT / "data" / "transcripts" / "transcripts.parquet"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models import HF_HUB_CACHE, MODEL_REGISTRY, list_models, load_model

print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA device count: {torch.cuda.device_count()}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU VRAM: {total_vram:.1f} GB")

print(f"Project root: {PROJECT_ROOT}")
print(f"Transcript parquet: {TRANSCRIPTS_PATH}")
print(f"Transcript parquet exists: {TRANSCRIPTS_PATH.exists()}")
print(f"Hugging Face model cache: {HF_HUB_CACHE}")
assert TRANSCRIPTS_PATH.exists(), "Transcript parquet not found. Run data/download_transcripts.py first."

Python: 3.12.7
Platform: Windows-11-10.0.26200-SP0
PyTorch: 2.12.0.dev20260322+cu128
Transformers: 4.57.6
CUDA available: True
USE_TF: 0
TRANSFORMERS_NO_TF: 1
CUDA device count: 1
GPU: NVIDIA GeForce RTX 5090 Laptop GPU
GPU VRAM: 23.9 GB
Project root: c:\Users\nyuss\OneDrive\Documentos\Bars\Portfollio\Books\NYU\Spring 2026\Text as Data\project
Transcript parquet: c:\Users\nyuss\OneDrive\Documentos\Bars\Portfollio\Books\NYU\Spring 2026\Text as Data\project\data\transcripts\transcripts.parquet
Transcript parquet exists: True
hf_xet installed: True


## 2. Model Registry

The model registry now lives in `src.models`. The notebook only selects which registered model to load.

In [11]:
# display registered models and choose the first Question 1 test model
SELECTED_MODEL_KEY = "qwen_2_5_14b"
assert SELECTED_MODEL_KEY in MODEL_REGISTRY, "Selected model key is not in MODEL_REGISTRY."

models_df = pd.DataFrame.from_dict(list_models(), orient="index")
display(models_df)

print(f"Selected model key: {SELECTED_MODEL_KEY}")
print(f"Selected Hugging Face ID: {MODEL_REGISTRY[SELECTED_MODEL_KEY].hf_id}")
print(f"Strategy order: {' -> '.join(MODEL_REGISTRY[SELECTED_MODEL_KEY].strategies)}")

Selected model key: qwen_2_5_14b
Selected Hugging Face ID: Qwen/Qwen2.5-14B-Instruct
Role: mid-sized instruction-tuned model; first smoke-test target


,hf_id,role
llama_3_1_8b,meta-llama/Llama-3.1-8B-Instruct,compact accessible baseline
qwen_2_5_14b,Qwen/Qwen2.5-14B-Instruct,mid-sized instruction-tuned model; first smoke...
deepseek_r1_distill_qwen_14b,deepseek-ai/DeepSeek-R1-Distill-Qwen-14B,reasoning-distilled model at similar scale to ...
mistral_small_3_1_24b,mistralai/Mistral-Small-3.1-24B-Instruct-2503,upper local capability tier


## 3. Transcript Loading & Schema Check

We use `structured_content`, not raw `content`, because the dataset already stores each call as ordered speaker turns.

In [4]:
# load transcript data and verify structured_content is available
tr = pd.read_parquet(TRANSCRIPTS_PATH)

print(f"Transcript rows: {len(tr):,}")
print(f"Columns: {list(tr.columns)}")
assert "structured_content" in tr.columns, "structured_content column is missing."

valid_structured = tr[tr["structured_content"].notna()].copy()
assert len(valid_structured) > 0, "No transcripts with structured_content found."

sample_row = valid_structured.iloc[0]
sample_transcript_id = sample_row.name
sample_turns = sample_row["structured_content"]

metadata_cols = ["symbol", "company_name", "date", "year", "quarter"]
print("Sample transcript metadata:")
display(sample_row[metadata_cols].to_frame("value"))

print(f"structured_content type: {type(sample_turns)}")
print(f"Number of speaker turns: {len(sample_turns):,}")
print()
print("First 4 speaker turns:")
for turn in sample_turns[:4]:
    speaker = str(turn.get("speaker", "")).strip()
    text = str(turn.get("text", "")).strip()
    print(f"Speaker: {speaker!r}")
    print(f"Text: {text[:160]}...")
    print()

Transcript rows: 33,362
Columns: ['symbol', 'quarter', 'year', 'date', 'content', 'structured_content', 'company_name', 'company_id']
Sample transcript metadata:


,value
symbol,A
company_name,"Agilent Technologies, Inc."
date,2020-11-23 16:30:00
year,2020
quarter,4


structured_content type: <class 'numpy.ndarray'>
Number of speaker turns: 113

First 4 speaker turns:
Speaker: 'Operator'
Text: Good afternoon, and welcome to the Agilent Technologies Fourth Quarter Earnings Conference Call. All lines have been placed on mute to prevent any background no...

Speaker: 'Ankur Dhingra'
Text: Thank you, and welcome everyone to Agilent's fourth quarter and full-year conference call for fiscal year 2020. With me are Mike McMullen, Agilent's President a...

Speaker: 'Mike McMullen'
Text: Thanks, Ankur, and thanks to everyone for joining us on our call today. Today, I want to get straight to our quarterly results, because they tell a very compell...

Speaker: 'Bob McMahon'
Text: Thanks, Mike, and good afternoon, everyone. In my remarks today, I'll provide some additional revenue detail, and take you through the fourth quarter income sta...



## 4. Turn-Preserving Batch Construction

Question 1 uses all 1,500-word non-operator, speaker-turn-preserving batches for one transcript. This avoids raw full-call prompts and avoids keyword filtering.

In [5]:
# build all 1,500-word non-operator batches for the selected transcript
MAX_BATCH_WORDS = 1500


def normalize_turn(turn):
    speaker = str(turn.get("speaker", "")).strip()
    text = str(turn.get("text", "")).strip()
    return speaker, text


def clean_structured_turns(structured_content):
    clean_turns = []
    for turn_id, turn in enumerate(structured_content):
        speaker, text = normalize_turn(turn)
        if speaker == "Operator" or text == "":
            continue
        clean_turns.append({
            "turn_id": turn_id,
            "speaker": speaker,
            "text": text,
            "word_count": len(text.split()),
        })
    return clean_turns


def build_turn_batches(clean_turns, max_words=1500):
    batches = []
    current_turns = []
    current_words = 0

    for turn in clean_turns:
        turn_words = turn["word_count"]
        if current_turns and current_words + turn_words > max_words:
            batches.append(current_turns)
            current_turns = []
            current_words = 0

        current_turns.append(turn)
        current_words += turn_words

    if current_turns:
        batches.append(current_turns)

    batch_rows = []
    for batch_id, turns in enumerate(batches):
        batch_text = "\n\n".join(
            f"{turn['speaker']}: {turn['text']}" if turn["speaker"] else turn["text"]
            for turn in turns
        )
        batch_rows.append({
            "transcript_id": sample_transcript_id,
            "batch_id": batch_id,
            "start_turn_id": turns[0]["turn_id"],
            "end_turn_id": turns[-1]["turn_id"],
            "n_turns": len(turns),
            "word_count": len(batch_text.split()),
            "batch_text": batch_text,
        })
    return pd.DataFrame(batch_rows)


clean_turns = clean_structured_turns(sample_turns)
assert len(clean_turns) > 0, "Selected transcript has no non-operator turns."

batches = build_turn_batches(clean_turns, max_words=MAX_BATCH_WORDS)
assert len(batches) > 0, "No batches were constructed."
assert batches["batch_text"].str.len().gt(0).all(), "At least one batch is empty."

print(f"Non-operator turns in transcript: {len(clean_turns):,}")
print(f"Batches constructed: {len(batches):,}")
display(batches[["batch_id", "start_turn_id", "end_turn_id", "n_turns", "word_count"]])

print("First batch preview:")
print(textwrap.shorten(batches.loc[0, "batch_text"].replace("\n", " "), width=1500, placeholder=" ..."))

Non-operator turns in transcript: 101
Turns in first batch: 2
First batch word count: 1,439
Start turn id: 1
End turn id: 2

Batch preview:
Ankur Dhingra: Thank you, and welcome everyone to Agilent's fourth quarter and full-year conference call for fiscal year 2020. With me are Mike McMullen, Agilent's President and CEO; and Bob McMahon, Agilent's Senior Vice President and CFO. Joining in the Q&A after Bob's comments will be: Jacob Thaysen, President of Agilent's Life Sciences & Applied Markets Group; Sam Raha, President of Agilent's Diagnostics and Genomics Group; and Padraig McDonnell, President of Agilent CrossLab Group. This presentation is being webcast live. The news release, Investor presentation, and information to supplement today's discussion along with the recording of this webcast are made available on our Web site at investor.agilent.com. Today's comments by Mike and Bob will refer to non-GAAP financial measures. You will find the most directly comparable GAAP financial me

## 5. Prompt Construction

The first research question asks whether a local open-source model can extract expected capital-spending direction from transcript text. Each batch gets the same JSON-constrained prompt.

In [6]:
# construct the Question 1 capex-direction prompt for any batch
CAPEX_QUESTION = "Based on this text only, how does the firm plan to change its capital spending over the next year?"

SYSTEM_MESSAGE = "You are a careful financial analyst extracting grounded corporate investment signals from earnings-call transcript text."


def build_capex_prompt(batch_text):
    prompt_parts = [
        "The following text is an excerpt from a company's earnings call transcript.",
        "",
        "Based on this text only, answer the following question:",
        CAPEX_QUESTION,
        "",
        "Choose exactly one choice:",
        "- increase substantially",
        "- increase",
        "- no change",
        "- decrease",
        "- decrease substantially",
        "- insufficient information",
        "",
        "Scoring map:",
        "- decrease substantially = -1",
        "- decrease = -0.5",
        "- no change = 0",
        "- increase = 0.5",
        "- increase substantially = 1",
        "- insufficient information = 0",
        "",
        "Important distinction:",
        "- Use \"no change\" only when the text discusses capital spending or investment policy and indicates stability.",
        "- Use \"insufficient information\" when the text does not provide enough information about future capital spending or investment policy.",
        "",
        "Return only valid JSON with this exact schema:",
        "{",
        "  \"choice\": \"increase substantially | increase | no change | decrease | decrease substantially | insufficient information\",",
        "  \"score\": -1 | -0.5 | 0 | 0.5 | 1,",
        "  \"information_state\": \"explicit_capex_guidance | implicit_investment_signal | neutral_no_material_change | insufficient_information\",",
        "  \"supporting_excerpt\": \"short quote from the provided text, or empty string if insufficient information\",",
        "  \"explanation\": \"one concise sentence explaining the choice\"",
        "}",
        "",
        "Transcript text:",
        "\"\"\"",
        batch_text,
        "\"\"\"",
    ]
    return "\n".join(prompt_parts).strip()


USER_PROMPT = build_capex_prompt(batches.loc[0, "batch_text"])
assert batches.loc[0, "batch_text"] in USER_PROMPT, "Prompt does not include batch text."
assert "Return only valid JSON" in USER_PROMPT, "Prompt does not include JSON instructions."
assert CAPEX_QUESTION in USER_PROMPT, "Prompt does not include the capex question."

print("System message:")
print(SYSTEM_MESSAGE)
print()
print("First batch prompt preview:")
print(textwrap.shorten(USER_PROMPT.replace("\n", " "), width=2500, placeholder=" ..."))

System message:
You are a careful financial analyst extracting grounded corporate investment signals from earnings-call transcript text.

User prompt preview:
The following text is an excerpt from a company's earnings call transcript. Based on this text only, answer the following question: Based on this text only, how does the firm plan to change its capital spending over the next year? Choose exactly one choice: - increase substantially - increase - no change - decrease - decrease substantially - insufficient information Scoring map: - decrease substantially = -1 - decrease = -0.5 - no change = 0 - increase = 0.5 - increase substantially = 1 - insufficient information = 0 Important distinction: - Use "no change" only when the text discusses capital spending or investment policy and indicates stability. - Use "insufficient information" when the text does not provide enough information about future capital spending or investment policy. Return only valid JSON with this exact schema: { "

## 6. Model Loading

Load the selected model through `src.models.load_model()`. The loader uses GPU-only placement and the least aggressive configured strategy: BF16 first where feasible, then 8-bit, then 4-bit NF4 only when needed.

In [12]:
# load selected model through the reusable loader
# This may download model files into the project-level models/ cache.
loaded = load_model(SELECTED_MODEL_KEY, device=0)

tokenizer = loaded["tokenizer"]
model = loaded["model"]

print(f"Loaded strategy: {loaded['strategy']}")
print(f"Model cache: {loaded['cache_dir']}")
print(f"Device map: {loaded['device_map']}")
print(f"VRAM: {loaded['vram']}")

Loading model: Qwen/Qwen2.5-14B-Instruct


c:\Users\nyuss\anaconda3\Lib\site-packages\transformers\modeling_utils.py:1984: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
`torch_dtype` is deprecated! Use `dtype` instead!


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

model-00002-of-00008.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

model-00005-of-00008.safetensors:   0%|          | 0.00/3.98G [00:00<?, ?B/s]

...

model-00006-of-00008.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

model-00008-of-00008.safetensors:   0%|          | 0.00/1.70G [00:00<?, ?B/s]

c:\Users\nyuss\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nyuss\.cache\huggingface\hub\models--Qwen--Qwen2.5-14B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

OSError: The paging file is too small for this operation to complete. (os error 1455)

## 7. Question 1 Batch Scoring

Score every batch for the selected transcript, parse JSON, and inspect the batch-level outputs before scaling beyond one transcript.

In [ ]:
# helper functions for generation and JSON parsing
def format_chat_prompt(system_message, user_prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_prompt},
    ]
    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template is not None:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    return f"System: {system_message}\n\nUser: {user_prompt}\n\nAssistant:"


def generate_response(user_prompt, max_new_tokens=500):
    prompt_text = format_chat_prompt(SYSTEM_MESSAGE, user_prompt)
    inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda:0")

    with torch.no_grad():
        generated = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = generated[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def parse_model_json(raw_output):
    try:
        return json.loads(raw_output), None
    except json.JSONDecodeError as exc:
        return None, str(exc)


score_rows = []

for _, batch in batches.iterrows():
    prompt = build_capex_prompt(batch["batch_text"])
    raw_output = generate_response(prompt)
    parsed, parse_error = parse_model_json(raw_output)

    row = batch.drop(labels=["batch_text"]).to_dict()
    row["raw_output"] = raw_output
    row["parse_error"] = parse_error

    if parsed is not None:
        row.update({
            "choice": parsed.get("choice"),
            "score": parsed.get("score"),
            "information_state": parsed.get("information_state"),
            "supporting_excerpt": parsed.get("supporting_excerpt"),
            "explanation": parsed.get("explanation"),
        })
    else:
        row.update({
            "choice": None,
            "score": None,
            "information_state": None,
            "supporting_excerpt": None,
            "explanation": None,
        })

    score_rows.append(row)
    print(f"Scored batch {int(batch['batch_id']) + 1}/{len(batches)} | choice={row['choice']} | score={row['score']}")

batch_scores = pd.DataFrame(score_rows)
display(batch_scores)

In [ ]:
# aggregate batch-level results to one transcript-level Question 1 score
valid_scores = batch_scores[batch_scores["parse_error"].isna()].copy()
valid_scores["score"] = pd.to_numeric(valid_scores["score"], errors="coerce")
valid_scores["is_informative"] = valid_scores["information_state"].ne("insufficient_information")

raw_mean_score = valid_scores["score"].mean()
informative_scores = valid_scores.loc[valid_scores["is_informative"], "score"]
informative_mean_score = informative_scores.mean() if len(informative_scores) else 0.0
share_informative = valid_scores["is_informative"].mean() if len(valid_scores) else 0.0

transcript_score = {
    "transcript_id": sample_transcript_id,
    "symbol": sample_row["symbol"],
    "company_name": sample_row["company_name"],
    "date": sample_row["date"],
    "model_key": SELECTED_MODEL_KEY,
    "model_id": loaded["model_id"],
    "loading_strategy": loaded["strategy"],
    "n_batches": len(batch_scores),
    "n_valid_batches": len(valid_scores),
    "n_informative_batches": int(valid_scores["is_informative"].sum()) if len(valid_scores) else 0,
    "share_informative": share_informative,
    "informative_mean_score": informative_mean_score,
    "raw_mean_score_diagnostic": raw_mean_score,
}

transcript_score = pd.Series(transcript_score).to_frame("value")
display(transcript_score)

## 8. First-Run Notes

After running Question 1 for one transcript, inspect:

- whether every batch returned parseable JSON;
- which batches were informative;
- whether supporting excerpts actually appear in the relevant batch text;
- whether `informative_mean_score` gives a sensible transcript-level direction;
- whether `share_informative` is low or high enough to trust the transcript-level score.

No persistent score files are written yet. Once one-transcript behavior is stable, the next notebook step is a small reproducible sample across several transcripts.